In [ ]:
import os, sys, json, shutil, subprocess, importlib
import numpy as np
import pandas as pd
import torch
from scipy.io import wavfile
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
try:
    from transformers import AutoFeatureExtractor, AutoModel
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"]); importlib.invalidate_caches()
    from transformers import AutoFeatureExtractor, AutoModel

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json"), "r") as f:
    config = json.load(f)
np.random.seed(config["seed"]); torch.manual_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR, FEATURES_DIR, RESULTS_DIR, WAV_DIR = (config["paths"][k] for k in ("data_dir", "features_dir", "results_dir", "audio_wav_dir"))
SPLITS = ["train", "dev", "test"]
AUDIO_DIM, SAMPLE_RATE = config["audio_feature_dim"], config["audio_sample_rate"]
MODEL_ID = config.get("audio_model", "microsoft/wavlm-base-plus")
MIN_SECONDS = config.get("audio_min_seconds", 0.1)
MAX_SECONDS = 60.0                      
MAX_MISSING = config["max_missing_fraction"]
N_WORKERS = max(1, min(4, os.cpu_count() or 1))
RECOMPUTE = os.environ.get("MELD_P4_RECOMPUTE", "0") == "1"
BACKBONE_TAG = f"{MODEL_ID}:uniform-mean-layers-1..last:mean-over-time"

assert str(config.get("audio_feature_set", "")).startswith("wavlm"), ("config.json still describes the old eGeMAPS setup -- re-run the PATCHED "
                                                                   "Phase 1 notebook first (it only changes the audio_*/video_* keys).")
manifests = {}
for s in SPLITS:
    m = pd.read_csv(os.path.join(DATA_DIR, f"media_manifest_{s}.csv"), keep_default_na=False)
    n_csv = len(pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv")))
    assert len(m) == n_csv and (m["row_idx"].values == np.arange(n_csv)).all(), f"[{s}] manifest/CSV misalignment"
    manifests[s] = m
    print(f"[{s}] manifest rows: {len(m)}  usable clips: {(m['status'] == 'ok').sum()}")
print(f"Device: {device} | model: {MODEL_ID} | dim {AUDIO_DIM} | min clip {MIN_SECONDS}s")

In [ ]:
def read_wav_mono_f32(path):
    """16-bit PCM (or float) wav -> (float32 mono in [-1,1], sample_rate)."""
    sr, x = wavfile.read(path)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / float(np.iinfo(x.dtype).max + 1)     
    else:
        x = x.astype(np.float32)
    if x.ndim > 1:
        x = x.mean(axis=1).astype(np.float32)
    return x, int(sr)


def check_wave(x, sr, expected_sr, min_seconds, max_seconds):
    """Returns (wave_or_None, reason_or_None, info). Too-short / wrong-rate / non-finite clips are UNUSABLE (no silent padding)."""
    info = {"truncated": False, "silent": False, "seconds": len(x) / float(sr) if sr else 0.0}
    if sr != expected_sr:
        return None, f"bad_sample_rate: {sr}", info
    if not np.isfinite(x).all():
        return None, "non_finite_samples", info
    if len(x) < int(round(min_seconds * sr)):
        return None, f"too_short: {len(x) / sr:.3f}s", info
    if len(x) > int(max_seconds * sr):
        x, info["truncated"] = x[: int(max_seconds * sr)], True
    info["silent"] = bool(np.abs(x).max() < 1e-4)
    return x, None, info


def pool_hidden_states(hs):
    """hs: list/array of L+1 arrays [T,D] (embedding output + L transformer layers) -> per-layer mean over time [L+1,D] (float32)."""
    return np.stack([np.asarray(h, dtype=np.float64).mean(axis=0) for h in hs]).astype(np.float32)


def utterance_feature(layer_means):
    """DECIDED rule: uniform mean of transformer layers 1..L (index 0 = embedding/CNN-projection output is excluded)."""
    return np.asarray(layer_means, dtype=np.float64)[1:].mean(axis=0).astype(np.float32)


def zscore_fit(x, valid):
    mu, sd = x[valid].mean(axis=0), x[valid].std(axis=0)
    return mu.astype(np.float32), np.where(sd < 1e-8, 1.0, sd).astype(np.float32)
print("Helpers defined.")

In [ ]:
extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()
for p_ in model.parameters():
    p_.requires_grad_(False)
N_HS = model.config.num_hidden_layers + 1                       
assert model.config.hidden_size == AUDIO_DIM, (f"model hidden size {model.config.hidden_size} != audio_feature_dim {AUDIO_DIM}: "
                                               f"set audio_feature_dim={model.config.hidden_size} in Phase 1 and re-run it")
assert int(getattr(extractor, "sampling_rate", SAMPLE_RATE)) == SAMPLE_RATE, "feature-extractor sampling rate != config"

@torch.no_grad()
def embed_wave(wave):
    """float32 mono wave -> per-layer time-mean [N_HS, D]. One clip per forward pass (no padding => no attention-mask issues)."""
    x = extractor(wave, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_values.to(device)
    hs = model(x, output_hidden_states=True).hidden_states
    assert len(hs) == N_HS
    return pool_hidden_states([h[0].float().cpu().numpy() for h in hs])

rs = np.random.RandomState(0)
w1, w2 = (rs.randn(SAMPLE_RATE).astype(np.float32) * 0.1 for _ in range(2))
a1, a1b, a2 = embed_wave(w1), embed_wave(w1), embed_wave(w2)
assert a1.shape == (N_HS, AUDIO_DIM) and np.isfinite(a1).all()
assert np.abs(a1 - a1b).max() < 1e-4, "embedding is not deterministic (model in train mode?)"
assert np.abs(a1 - a2).mean() > 1e-4, "different inputs give identical embeddings"
print(f"Self-check OK: {N_HS} hidden states x {AUDIO_DIM}-d, deterministic. Utterance feature = uniform mean of layers 1..{N_HS - 1}.")

In [ ]:
def extract_wav(job):
    """Returns None on success, otherwise a short failure reason."""
    video_path, wav_path = job
    if os.path.exists(wav_path) and os.path.getsize(wav_path) > 44:
        return None
    os.makedirs(os.path.dirname(wav_path), exist_ok=True)
    tmp = wav_path + ".part.wav"
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", video_path,
           "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE), "-acodec", "pcm_s16le", tmp]
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    except subprocess.TimeoutExpired:
        return "ffmpeg_timeout"
    if p.returncode != 0 or not os.path.exists(tmp) or os.path.getsize(tmp) <= 44:
        if os.path.exists(tmp):
            os.remove(tmp)
        return "ffmpeg_failed: " + (p.stderr.strip().replace("\n", " ")[:200] or "no audio stream / empty output")
    os.replace(tmp, wav_path)
    return None

def wav_path_for(split, video_file):
    return os.path.join(WAV_DIR, split, os.path.splitext(video_file)[0] + ".wav")
print("ffmpeg extraction helper defined.")

In [ ]:
META_PATH = os.path.join(FEATURES_DIR, "audio_features_meta.json")
BACKUP_DIR = os.path.join(FEATURES_DIR, "_backup_egemaps")
META = {"backbone": BACKBONE_TAG, "dim": AUDIO_DIM, "n_hidden_states": N_HS, "min_seconds": MIN_SECONDS, "max_seconds": MAX_SECONDS}

def backup_legacy():
    names = [f"audio_features_raw_{s}.npy" for s in SPLITS] + [f"audio_features_{s}.npy" for s in SPLITS] + \
            [f"audio_valid_{s}.npy" for s in SPLITS] + ["audio_scaler.npz"]
    present = [f for f in names if os.path.exists(os.path.join(FEATURES_DIR, f))]
    if not present:
        return
    os.makedirs(BACKUP_DIR, exist_ok=True)
    for f in present:
        dst = os.path.join(BACKUP_DIR, f)
        shutil.move(os.path.join(FEATURES_DIR, f), dst + (".dup" if os.path.exists(dst) else ""))
    for s in SPLITS:
        r = os.path.join(RESULTS_DIR, f"audio_extraction_report_{s}.json")
        if os.path.exists(r) and not os.path.exists(os.path.join(BACKUP_DIR, os.path.basename(r))):
            shutil.copy2(r, os.path.join(BACKUP_DIR, os.path.basename(r)))
    print(f"Moved {len(present)} legacy audio cache files to {BACKUP_DIR}")

def process_split(split):
    raw_path = os.path.join(FEATURES_DIR, f"audio_features_raw_{split}.npy")
    layers_path = os.path.join(FEATURES_DIR, f"audio_layers_{split}.npy")
    valid_path = os.path.join(FEATURES_DIR, f"audio_valid_{split}.npy")
    if not RECOMPUTE and all(os.path.exists(p) for p in (raw_path, layers_path, valid_path)):     
        raw, valid = np.load(raw_path), np.load(valid_path)
        print(f"[{split}] features already complete -> skipping (unusable: {1.0 - valid.mean():.3%}).")
        return raw, valid
    m = manifests[split]; n = len(m)
    raw = np.zeros((n, AUDIO_DIM), np.float32); layers = np.zeros((n, N_HS, AUDIO_DIM), np.float16); valid = np.zeros(n, bool)
    reasons = {int(i): f"video_{st}" for i, st in zip(m["row_idx"], m["status"]) if st != "ok"}
    secs, n_trunc, n_silent = [], 0, 0

    ok_rows = m[m["status"] == "ok"]
    jobs = [(vp, wav_path_for(split, vf)) for vp, vf in zip(ok_rows["video_path"], ok_rows["video_file"])]
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = list(tqdm(ex.map(extract_wav, jobs), total=len(jobs), desc=f"ffmpeg audio ({split})"))
    for i, res in zip(ok_rows["row_idx"], results):
        if res is not None:
            reasons[int(i)] = res

    for i in tqdm(range(n), desc=f"WavLM ({split})"):
        if i in reasons:
            continue
        try:
            x, sr = read_wav_mono_f32(wav_path_for(split, m.at[i, "video_file"]))
            wave, why, info = check_wave(x, sr, SAMPLE_RATE, MIN_SECONDS, MAX_SECONDS)
            if why is not None:
                reasons[i] = why
                continue
            lm = embed_wave(wave)
        except Exception as e:
            reasons[i] = f"wavlm_failed: {type(e).__name__}: {str(e)[:150]}"
            continue
        feat = utterance_feature(lm)
        if not (np.isfinite(feat).all() and np.isfinite(lm).all()):
            reasons[i] = "non_finite_features"
            continue
        raw[i], layers[i], valid[i] = feat, lm.astype(np.float16), True
        secs.append(info["seconds"]); n_trunc += int(info["truncated"]); n_silent += int(info["silent"])

    failures = [{"row_idx": i, "Dialogue_ID": int(m.at[i, "Dialogue_ID"]), "Utterance_ID": int(m.at[i, "Utterance_ID"]), "reason": r}
                for i, r in sorted(reasons.items())]
    frac = len(failures) / n
    sec = np.asarray(secs) if secs else np.zeros(1)
    with open(os.path.join(RESULTS_DIR, f"audio_extraction_report_{split}.json"), "w") as f:
        json.dump({"split": split, "backbone": BACKBONE_TAG, "n": n, "n_valid": int(valid.sum()), "n_failed": len(failures), "fraction_failed": frac,
                   "n_truncated": n_trunc, "n_silent_but_valid": n_silent,
                   "seconds_mean": float(sec.mean()), "seconds_p95": float(np.percentile(sec, 95)), "seconds_max": float(sec.max()),
                   "failures": failures}, f, indent=2)
    print(f"[{split}] valid: {int(valid.sum())}/{n}  failed: {len(failures)} ({frac:.3%}) | truncated {n_trunc} | silent-but-valid {n_silent} | mean {sec.mean():.2f}s")
    if frac > MAX_MISSING:
        raise RuntimeError(f"[{split}] {len(failures)} unusable audio clips ({frac:.2%}) exceed the allowed {MAX_MISSING:.2%}. "
                           f"See results/audio_extraction_report_{split}.json")
    np.save(layers_path, layers); np.save(raw_path, raw); np.save(valid_path, valid)
    return raw, valid

prev_meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else None
if prev_meta != META and not RECOMPUTE:
    backup_legacy()                                
with open(META_PATH, "w") as f:                     
    json.dump(META, f, indent=2)
raw_feats, valid_masks = {}, {}
for s in SPLITS:
    raw_feats[s], valid_masks[s] = process_split(s)

In [ ]:
train_valid_rows = raw_feats["train"][valid_masks["train"]]
assert len(train_valid_rows) > 100, "Too few valid train rows to fit the scaler."
mean, std_safe = zscore_fit(raw_feats["train"], valid_masks["train"])
constant = train_valid_rows.std(axis=0) < 1e-8
print(f"Scaler fitted on {len(train_valid_rows)} valid train clips; constant features: {int(constant.sum())}")
for s in SPLITS:
    z = ((raw_feats[s] - mean) / std_safe).astype(np.float32)
    z[~valid_masks[s]] = 0.0     
    np.save(os.path.join(FEATURES_DIR, f"audio_features_{s}.npy"), z)
    print(f"Saved audio_features_{s}.npy with shape {z.shape}")
np.savez(os.path.join(FEATURES_DIR, "audio_scaler.npz"), mean=mean, std=std_safe, n_fit=len(train_valid_rows))

In [ ]:
for s in SPLITS:
    z = np.load(os.path.join(FEATURES_DIR, f"audio_features_{s}.npy"))
    v = np.load(os.path.join(FEATURES_DIR, f"audio_valid_{s}.npy"))
    n_csv = len(pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv")))
    assert z.shape == (n_csv, AUDIO_DIM), f"[{s}] bad shape {z.shape}"
    assert np.isfinite(z).all(), f"[{s}] non-finite values"
    assert (z[~v] == 0).all()
    assert z[v].std(axis=0).mean() > 1e-3, f"[{s}] audio features are (near-)constant -- extraction is not working"
    print(f"[{s}] OK: {int(v.sum())}/{n_csv} valid ({v.mean():.2%})")
zt = np.load(os.path.join(FEATURES_DIR, "audio_features_train.npy"))[np.load(os.path.join(FEATURES_DIR, "audio_valid_train.npy"))]
nonconst = ~constant
assert np.abs(zt.mean(axis=0)[nonconst]).max() < 1e-3, "train z-score mean is not ~0"
assert np.abs(zt.std(axis=0)[nonconst] - 1).max() < 1e-3, "train z-score std is not ~1"
print("Phase 4 Audio Feature Extraction (WavLM) completed. Next: Step 3 (downstream plumbing) -- do NOT start Phase 6+ yet; Phase 6 still hard-codes the old dims.")